
# V30 OFFICIAL — mE5 + Twitter-XLM-R Research Ensemble

**Target: Kaggle ≥ 0.600**

V29 did **not** pass the gate. V30 follows the strongest recent Bangla hate-speech research more
closely instead of training more copies of the same models.

Research from BLP-2025 reports strong results from:

- **multilingual-e5-base**
- **Twitter XLM-R**
- BanglaBERT / MuRIL
- layer-wise learning-rate decay (LLRD)
- adversarial fine-tuning
- logit/probability ensembling

V30 therefore trains only **two new complementary encoders**:

1. `intfloat/multilingual-e5-base` — semantic representation
2. `cardiffnlp/twitter-xlm-roberta-base` — multilingual social-media representation

It then combines them with the already-computed V29 OOF signals and a regularized meta-stacker.

## Additional signal

V30 also evaluates the **original pretrained binary abusive heads** from:

- BanglaHateBERT
- Bengali-Abusive MuRIL

These heads are used as frozen toxicity signals instead of replacing their learned abusive-language
classifier weights.

## Important

- Correct official split only: 3292 / 500 / 1000.
- Neural models are trained only on `train.csv`.
- Validation is used for final model/calibration selection, which the competition explicitly allows.
- No manual test labeling.
- No external labeled dataset is added in this version.
- Completed transformer folds are cached.

Send `v30_results.zip` to ChatGPT before submitting.


In [ ]:

# 1. Upload V30 data
from google.colab import files
import pathlib, zipfile

uploaded = files.upload()
assert "v30_OFFICIAL_data.zip" in uploaded, "Upload v30_OFFICIAL_data.zip."

DATA_DIR = pathlib.Path("/content/v30_data")
DATA_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile("v30_OFFICIAL_data.zip") as z:
    z.extractall(DATA_DIR)

print("Extracted:")
for p in sorted(DATA_DIR.iterdir()):
    print(" ", p.name, f"{p.stat().st_size:,} bytes")


In [ ]:

# 2. Install + checkpoint preflight
import sys, subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.6",
    "tokenizers==0.22.1",
    "sentencepiece==0.2.1",
    "scikit-learn",
    "scipy",
    "pandas",
    "numpy",
    "tqdm",
    "safetensors",
])

from transformers import AutoConfig, AutoTokenizer

CHECKPOINTS = {
    "me5": "intfloat/multilingual-e5-base",
    "twitter": "cardiffnlp/twitter-xlm-roberta-base",
    "banglahate_binary": "saroarj/BanglaHateBert",
    "muril_binary": "Hate-speech-CNERG/bengali-abusive-MuRIL",
}

for k,v in CHECKPOINTS.items():
    print("Checking",k,v)
    _ = AutoConfig.from_pretrained(v)
    _ = AutoTokenizer.from_pretrained(v)

print("✅ All V30 checkpoints available.")


In [ ]:

# 3. Imports, official data, preprocessing
import gc, re, math, json, random, zipfile, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

SEED=2026
N_FOLDS=5
EPOCHS=3
SNAPSHOT_EPOCHS=[2,3]
MAX_LENGTH=128
BATCH=8
ACCUM=2
EVAL_BATCH=32
BASE_LR=1.2e-5
HEAD_LR=3e-5
LAYER_DECAY=0.90
WEIGHT_DECAY=0.01
WARMUP_RATIO=0.10
FGM_EPS=0.25

LABELS=[
    "is_toxic","implicit_toxicity","sarcasm","contains_profanity",
    "threatening","identity_attack","sexual_content"
]

CHECKPOINTS={
    "me5":"intfloat/multilingual-e5-base",
    "twitter":"cardiffnlp/twitter-xlm-roberta-base",
}

RUN_DIR=Path("/content/v30_output")
RUN_DIR.mkdir(exist_ok=True)

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False

seed_all(SEED)
assert torch.cuda.is_available(),"Enable T4 GPU."

train=pd.read_csv(DATA_DIR/"train.csv",dtype={"ID":str})
valid=pd.read_csv(DATA_DIR/"validation.csv",dtype={"ID":str})
test=pd.read_csv(DATA_DIR/"test.csv",dtype={"ID":str})
sample=pd.read_csv(DATA_DIR/"sample_submission.csv",dtype={"ID":str})

assert len(train)==3292 and len(valid)==500 and len(test)==1000
assert test.ID.tolist()==sample.ID.tolist()
assert set(train.ID).isdisjoint(valid.ID)
assert set(train.ID).isdisjoint(test.ID)
assert set(valid.ID).isdisjoint(test.ID)

v29=np.load(DATA_DIR/"v29_probabilities.npz")

def clean_text(x):
    x=unicodedata.normalize("NFKC",str(x))
    x=x.replace("\u200c"," ").replace("\u200d"," ")
    return re.sub(r"\s+"," ",x).strip()

def twitter_clean(x):
    out=[]
    for t in clean_text(x).split():
        if t.startswith("@") and len(t)>1:
            out.append("@user")
        elif t.startswith("http"):
            out.append("http")
        else:
            out.append(t)
    return " ".join(out)

def group_key(x):
    return clean_text(x).lower()

for df in [train,valid,test]:
    df["text_clean"]=df.text.map(clean_text)
    df["twitter_text"]=df.text.map(twitter_clean)
    df["me5_text"]="query: "+df["text_clean"]
    df["group_key"]=df.text.map(group_key)

print("✅ official split verified")
display(pd.DataFrame({
    "train":train[LABELS].mean(),
    "validation":valid[LABELS].mean()
}).round(4))


In [ ]:

# 4. Rebuild the same grouped 5-fold TRAIN split used for V29
groups=train.groupby("group_key",sort=False).agg(
    size=("ID","size"),
    **{f"sum_{l}":(l,"sum") for l in LABELS}
).reset_index()

global_pos=np.array([train[l].sum() for l in LABELS],float)
total=float(len(train))
rarity=1/np.sqrt(np.maximum(global_pos,1))
mat=groups[[f"sum_{l}" for l in LABELS]].to_numpy(float)

rng=np.random.default_rng(SEED)
priority=(mat*rarity).sum(1)+0.005*groups["size"].to_numpy()
order=np.argsort(-(priority+rng.random(len(groups))*1e-8))

fp=np.zeros((N_FOLDS,len(LABELS)))
fs=np.zeros(N_FOLDS)
assign={}

for idx in order:
    row=groups.iloc[idx]
    vec=np.array([row[f"sum_{l}"] for l in LABELS],float)
    size=float(row["size"])

    costs=[]
    for f in range(N_FOLDS):
        p=fp.copy()
        s=fs.copy()
        p[f]+=vec
        s[f]+=size

        costs.append(
            np.mean((p/np.maximum(global_pos[None,:],1)-1/N_FOLDS)**2)
            +0.30*np.mean((s/total-1/N_FOLDS)**2)
        )

    best=int(np.argmin(costs))
    assign[row["group_key"]]=best
    fp[best]+=vec
    fs[best]+=size

train["fold"]=train.group_key.map(assign).astype(int)

assert train.groupby("group_key").fold.nunique().max()==1
assert set(train.fold)==set(range(5))

print(train.groupby("fold").size().to_dict())
display(train.groupby("fold")[LABELS].sum())


In [ ]:

# 5. Hierarchical classifier + loss + FGM + LLRD
class TextDS(Dataset):
    def __init__(self,df,tok,text_col,labels=True):
        self.text=df[text_col].astype(str).tolist()
        self.y=df[LABELS].to_numpy(np.float32) if labels else None
        self.tok=tok

    def __len__(self):
        return len(self.text)

    def __getitem__(self,i):
        e=self.tok(
            self.text[i],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
        )
        if self.y is not None:
            e["labels"]=self.y[i]
        return e

class Coll:
    def __init__(self,tok):
        self.tok=tok

    def __call__(self,items):
        y=None
        if "labels" in items[0]:
            y=torch.tensor(
                np.stack([x.pop("labels") for x in items]),
                dtype=torch.float32
            )

        b=self.tok.pad(
            items,padding=True,pad_to_multiple_of=8,return_tensors="pt"
        )
        if y is not None:
            b["labels"]=y
        return b

class HierClassifier(nn.Module):
    def __init__(self,name,vocab_size):
        super().__init__()
        self.base=AutoModel.from_pretrained(name)
        if self.base.get_input_embeddings().num_embeddings!=vocab_size:
            self.base.resize_token_embeddings(vocab_size)

        h=self.base.config.hidden_size
        self.dropout=nn.Dropout(0.15)
        self.toxic=nn.Linear(h*2,1)

        # Attribute head receives representation + predicted-toxic logit.
        self.attrs=nn.Sequential(
            nn.Linear(h*2+1,h),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(h,6),
        )

    def forward(self,input_ids,attention_mask,token_type_ids=None):
        kw={"input_ids":input_ids,"attention_mask":attention_mask}
        if token_type_ids is not None:
            kw["token_type_ids"]=token_type_ids

        out=self.base(**kw)
        hs=out.last_hidden_state

        cls=hs[:,0]
        mask=attention_mask.unsqueeze(-1).float()
        mean=(hs*mask).sum(1)/mask.sum(1).clamp_min(1.0)

        rep=self.dropout(torch.cat([cls,mean],dim=-1))
        tox=self.toxic(rep)
        attrs=self.attrs(torch.cat([rep,tox.detach()],dim=-1))

        return torch.cat([tox,attrs],dim=1)

def hierarchical_loss(logits,y):
    # Toxicity: all examples.
    tox=torch.nn.functional.binary_cross_entropy_with_logits(
        logits[:,0],y[:,0],reduction="none"
    )

    attr_raw=torch.nn.functional.binary_cross_entropy_with_logits(
        logits[:,1:],y[:,1:],reduction="none"
    )

    toxic=y[:,0:1]

    # implicit/sarcasm can exist outside toxic label, so retain all negatives.
    weights=torch.ones_like(attr_raw)

    # Explicit attributes: non-toxic negatives are easy; downweight them.
    explicit_easy=torch.tensor(
        [1.0,1.0,0.12,0.12,0.12,0.12],
        device=logits.device
    ).unsqueeze(0)

    neg_weight=toxic+(1-toxic)*explicit_easy
    weights=torch.where(y[:,1:]>0.5,torch.ones_like(weights),neg_weight)

    # Mild positive weighting only.
    pos_boost=torch.tensor(
        [1.4,1.4,1.2,1.5,1.25,1.5],
        device=logits.device
    ).unsqueeze(0)
    weights=torch.where(y[:,1:]>0.5,weights*pos_boost,weights)

    attr=(attr_raw*weights).sum(1)/weights.sum(1).clamp_min(1e-6)

    # Match competition weighting.
    return (0.5*tox+0.5*attr).mean()

class FGM:
    def __init__(self,model,eps=FGM_EPS):
        self.model=model
        self.eps=eps
        self.backup={}

    def attack(self):
        for n,p in self.model.named_parameters():
            if (
                p.requires_grad
                and p.grad is not None
                and "word_embeddings" in n
            ):
                norm=torch.norm(p.grad)
                if torch.isfinite(norm) and norm>0:
                    self.backup[n]=p.data.clone()
                    p.data.add_(self.eps*p.grad/norm)

    def restore(self):
        for n,p in self.model.named_parameters():
            if n in self.backup:
                p.data=self.backup[n]
        self.backup={}

def llrd_groups(model):
    n_layers=int(getattr(model.base.config,"num_hidden_layers",12))
    groups={}

    for name,p in model.named_parameters():
        if not p.requires_grad:
            continue

        if name.startswith("toxic") or name.startswith("attrs"):
            lr=HEAD_LR
        else:
            m=re.search(r"encoder\.layer\.(\d+)",name)
            if m:
                layer=int(m.group(1))
                lr=BASE_LR*(LAYER_DECAY**(n_layers-1-layer))
            else:
                lr=BASE_LR*(LAYER_DECAY**n_layers)

        no_decay=(
            name.endswith("bias")
            or "LayerNorm.weight" in name
            or "layer_norm.weight" in name
        )
        wd=0.0 if no_decay else WEIGHT_DECAY
        groups.setdefault((lr,wd),[]).append(p)

    return [
        {"params":ps,"lr":lr,"weight_decay":wd}
        for (lr,wd),ps in groups.items()
    ]

@torch.no_grad()
def predict(model,loader):
    model.eval()
    out=[]

    for b in loader:
        b.pop("labels",None)
        inp={k:v.to("cuda",non_blocking=True) for k,v in b.items()}

        with torch.autocast(device_type="cuda",dtype=torch.float16):
            logits=model(**inp)

        out.append(torch.sigmoid(logits.float()).cpu().numpy())

    return np.concatenate(out)


In [ ]:

# 6. Train mE5 / Twitter-XLM-R 5-fold OOF
fold_seeds=[2026,17,3407,1337,42]

def train_new_expert(expert):
    name=CHECKPOINTS[expert]
    text_col="me5_text" if expert=="me5" else "twitter_text"

    oof=np.zeros((len(train),7),np.float32)
    val_parts=[]
    test_parts=[]
    hist=[]

    tok=AutoTokenizer.from_pretrained(name)

    for fold in range(5):
        tr=train[train.fold!=fold].copy()
        va=train[train.fold==fold].copy()

        cache=RUN_DIR/f"{expert}_fold_{fold}.npz"
        hp=RUN_DIR/f"{expert}_fold_{fold}_history.csv"

        if cache.exists():
            with np.load(cache) as z:
                idx=z["idx"]
                pv=z["valid"]
                pe=z["external_valid"]
                pt=z["test"]

            if np.array_equal(idx,va.index.to_numpy()):
                print("✅ reuse",expert,fold)
                oof[idx]=pv
                val_parts.append(pe)
                test_parts.append(pt)
                continue

        print("\n"+"="*72)
        print(expert,"fold",fold)

        seed_all(fold_seeds[fold])
        gc.collect()
        torch.cuda.empty_cache()

        col=Coll(tok)

        tl=DataLoader(
            TextDS(tr,tok,text_col,True),
            batch_size=BATCH,shuffle=True,num_workers=0,pin_memory=True,
            collate_fn=col
        )
        vl=DataLoader(
            TextDS(va,tok,text_col,True),
            batch_size=EVAL_BATCH,shuffle=False,num_workers=0,pin_memory=True,
            collate_fn=col
        )
        evl=DataLoader(
            TextDS(valid,tok,text_col,False),
            batch_size=EVAL_BATCH,shuffle=False,num_workers=0,pin_memory=True,
            collate_fn=col
        )
        tel=DataLoader(
            TextDS(test,tok,text_col,False),
            batch_size=EVAL_BATCH,shuffle=False,num_workers=0,pin_memory=True,
            collate_fn=col
        )

        model=HierClassifier(name,len(tok)).to("cuda")
        opt=torch.optim.AdamW(llrd_groups(model))

        steps=math.ceil(len(tl)/ACCUM)*EPOCHS
        sched=get_linear_schedule_with_warmup(
            opt,int(steps*WARMUP_RATIO),steps
        )
        scaler=torch.amp.GradScaler("cuda")
        fgm=FGM(model)

        sva=[]
        sev=[]
        ste=[]
        fh=[]

        for epoch in range(1,EPOCHS+1):
            model.train()
            opt.zero_grad(set_to_none=True)

            running=0.0
            seen=0

            for step,b in enumerate(
                tqdm(tl,desc=f"{expert} f{fold} e{epoch}/{EPOCHS}")
            ):
                y=b.pop("labels").to("cuda",non_blocking=True)
                inp={k:v.to("cuda",non_blocking=True) for k,v in b.items()}

                with torch.autocast(device_type="cuda",dtype=torch.float16):
                    logits=model(**inp)
                    loss=hierarchical_loss(logits,y)

                scaler.scale(loss/ACCUM).backward()

                # Light adversarial pass in epochs 2-3.
                # Do NOT unscale here; FGM normalizes the common AMP scale.
                if epoch>=2 and step%2==0:
                    fgm.attack()

                    with torch.autocast(device_type="cuda",dtype=torch.float16):
                        adv_logits=model(**inp)
                        adv_loss=hierarchical_loss(adv_logits,y)

                    fgm.restore()
                    scaler.scale((0.35*adv_loss)/ACCUM).backward()

                running+=float(loss.item())*len(y)
                seen+=len(y)

                if (step+1)%ACCUM==0 or step+1==len(tl):
                    # exactly ONE unscale per optimizer update
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)

                    old=scaler.get_scale()
                    scaler.step(opt)
                    scaler.update()

                    if scaler.get_scale()>=old:
                        sched.step()

                    opt.zero_grad(set_to_none=True)

            rec={
                "expert":expert,"fold":fold,"epoch":epoch,
                "loss":running/max(seen,1)
            }
            print(rec)
            fh.append(rec)
            hist.append(rec)
            pd.DataFrame(fh).to_csv(hp,index=False)

            if epoch in SNAPSHOT_EPOCHS:
                sva.append(predict(model,vl))
                sev.append(predict(model,evl))
                ste.append(predict(model,tel))

        pv=np.mean(sva,axis=0)
        pe=np.mean(sev,axis=0)
        pt=np.mean(ste,axis=0)

        oof[va.index.to_numpy()]=pv
        val_parts.append(pe)
        test_parts.append(pt)

        np.savez_compressed(
            cache,
            idx=va.index.to_numpy(),
            valid=pv,
            external_valid=pe,
            test=pt,
        )

        del model,opt,sched,scaler,fgm,tl,vl,evl,tel
        gc.collect()
        torch.cuda.empty_cache()

    pval=np.mean(val_parts,axis=0)
    ptest=np.mean(test_parts,axis=0)

    np.savez_compressed(
        RUN_DIR/f"{expert}_oof_valid_test.npz",
        oof=oof,validation=pval,test=ptest
    )

    pd.DataFrame(hist).to_csv(
        RUN_DIR/f"{expert}_training_history.csv",index=False
    )

    return oof,pval,ptest


In [ ]:

# 7. Train the two new research-backed encoders
me5_oof,me5_val,me5_test=train_new_expert("me5")
tw_oof,tw_val,tw_test=train_new_expert("twitter")

print("✅ mE5 + Twitter-XLM-R complete.")


In [ ]:

# 8. Frozen original abusive-head probabilities
# These keep the classifiers learned during external abusive-language pretraining.

@torch.no_grad()
def binary_checkpoint_probs(model_name,text_col):
    tok=AutoTokenizer.from_pretrained(model_name)
    model=AutoModelForSequenceClassification.from_pretrained(model_name).to("cuda")
    model.eval()

    def run(df):
        ds=TextDS(df,tok,text_col,False)
        dl=DataLoader(
            ds,batch_size=32,shuffle=False,num_workers=0,
            pin_memory=True,collate_fn=Coll(tok)
        )
        out=[]
        for b in dl:
            inp={k:v.to("cuda") for k,v in b.items()}
            with torch.autocast(device_type="cuda",dtype=torch.float16):
                logits=model(**inp).logits.float()

            if logits.shape[1]==1:
                p=torch.sigmoid(logits[:,0])
            else:
                p=torch.softmax(logits,dim=1)[:,1]

            out.append(p.cpu().numpy())

        return np.concatenate(out)

    a=run(train)
    b=run(valid)
    c=run(test)

    del model
    gc.collect()
    torch.cuda.empty_cache()

    return a,b,c

bh0_train,bh0_val,bh0_test=binary_checkpoint_probs(
    "saroarj/BanglaHateBert","text_clean"
)
mu0_train,mu0_val,mu0_test=binary_checkpoint_probs(
    "Hate-speech-CNERG/bengali-abusive-MuRIL","text_clean"
)

# Determine orientation on official training labels only.
yt=train.is_toxic.to_numpy(int)

def orient(p,y):
    try:
        a=roc_auc_score(y,p)
    except Exception:
        a=0.5

    if a<0.5:
        return 1-p,1-a,True

    return p,a,False

bh0_train,bh_auc,bh_flip=orient(bh0_train,yt)
mu0_train,mu_auc,mu_flip=orient(mu0_train,yt)

if bh_flip:
    bh0_val=1-bh0_val
    bh0_test=1-bh0_test

if mu_flip:
    mu0_val=1-mu0_val
    mu0_test=1-mu0_test

print("Frozen abusive-head train AUC:")
print("BanglaHateBERT:",bh_auc,"flipped:",bh_flip)
print("Abusive-MuRIL:",mu_auc,"flipped:",mu_flip)

np.savez_compressed(
    RUN_DIR/"frozen_abuse_heads.npz",
    bh_train=bh0_train,bh_validation=bh0_val,bh_test=bh0_test,
    mu_train=mu0_train,mu_validation=mu0_val,mu_test=mu0_test,
)


In [ ]:

# 9. Competition metric helpers
y_train=train[LABELS].to_numpy(np.int8)
y_valid=valid[LABELS].to_numpy(np.int8)
folds=train.fold.to_numpy(int)

def f1_pos(y,p):
    y=np.asarray(y,np.int8)
    p=np.asarray(p,np.int8)
    tp=np.sum((y==1)&(p==1))
    fp=np.sum((y==0)&(p==1))
    fn=np.sum((y==1)&(p==0))
    d=2*tp+fp+fn
    return float(2*tp/d) if d else 0.0

def f1_macro(y,p):
    vals=[]
    for c in [0,1]:
        tp=np.sum((y==c)&(p==c))
        fp=np.sum((y!=c)&(p==c))
        fn=np.sum((y==c)&(p!=c))
        d=2*tp+fp+fn
        vals.append(2*tp/d if d else 0.0)
    return float(np.mean(vals))

def metric(label,y,p):
    return f1_macro(y,p) if label=="is_toxic" else f1_pos(y,p)

def official(y,p):
    scores=[metric(l,y[:,j],p[:,j]) for j,l in enumerate(LABELS)]
    return float(.5*scores[0]+.5*np.mean(scores[1:])),scores

def topk_pred(p,rate):
    k=int(round(len(p)*rate))
    k=max(0,min(k,len(p)))
    out=np.zeros(len(p),np.int8)
    if k:
        out[np.argsort(-p)[:k]]=1
    return out

def best_threshold(label,y,p):
    vals=np.unique(p)
    if len(vals)>500:
        vals=np.quantile(vals,np.linspace(0,1,500))

    best=(-1,0.5)

    for t in vals:
        pred=(p>=t).astype(np.int8)
        s=metric(label,y,pred)
        if s>best[0]:
            best=(s,float(t))

    return best[1]


In [ ]:

# 10. Regularized OOF stackers
# Existing V29 signals + new mE5/Twitter signals.

old_sources = {
    "lex":("lexical_oof","lexical_validation","lexical_test"),
    "bh29":("banglahate_oof","banglahate_validation","banglahate_test"),
    "mu29":("abusive_muril_oof","abusive_muril_validation","abusive_muril_test"),
    "xl29":("xlmr_oof","xlmr_validation","xlmr_test"),
    "bg29":("banglish_oof","banglish_validation","banglish_test"),
}

def build_features(j,split):
    if split=="train":
        cols=[
            me5_oof[:,j],tw_oof[:,j],
            v29["lexical_oof"][:,j],
            v29["banglahate_oof"][:,j],
            v29["abusive_muril_oof"][:,j],
            v29["xlmr_oof"][:,j],
            v29["banglish_oof"][:,j],
        ]
        if j==0:
            cols += [bh0_train,mu0_train]

    elif split=="validation":
        cols=[
            me5_val[:,j],tw_val[:,j],
            v29["lexical_validation"][:,j],
            v29["banglahate_validation"][:,j],
            v29["abusive_muril_validation"][:,j],
            v29["xlmr_validation"][:,j],
            v29["banglish_validation"][:,j],
        ]
        if j==0:
            cols += [bh0_val,mu0_val]

    else:
        cols=[
            me5_test[:,j],tw_test[:,j],
            v29["lexical_test"][:,j],
            v29["banglahate_test"][:,j],
            v29["abusive_muril_test"][:,j],
            v29["xlmr_test"][:,j],
            v29["banglish_test"][:,j],
        ]
        if j==0:
            cols += [bh0_test,mu0_test]

    # logits improve linear separability; raw probs preserve calibration.
    raw=np.column_stack(cols)
    lg=np.log(np.clip(raw,1e-6,1-1e-6)/(1-np.clip(raw,1e-6,1-1e-6)))

    return np.column_stack([raw,lg])

stack_oof=np.zeros((len(train),7),float)
stack_val=np.zeros((len(valid),7),float)
stack_test=np.zeros((len(test),7),float)

stack_report=[]

for j,label in enumerate(LABELS):
    X=build_features(j,"train")
    Xv=build_features(j,"validation")
    Xt=build_features(j,"test")
    yy=y_train[:,j]

    # Meta OOF prediction.
    meta_oof=np.zeros(len(train),float)

    for f in range(5):
        held=folds==f

        pipe=Pipeline([
            ("scale",StandardScaler()),
            ("lr",LogisticRegression(
                C=0.10,
                solver="liblinear",
                max_iter=1000,
                random_state=SEED+f,
            ))
        ])
        pipe.fit(X[~held],yy[~held])
        meta_oof[held]=pipe.predict_proba(X[held])[:,1]

    # Final meta model for validation/test.
    final=Pipeline([
        ("scale",StandardScaler()),
        ("lr",LogisticRegression(
            C=0.10,
            solver="liblinear",
            max_iter=1000,
            random_state=SEED,
        ))
    ])
    final.fit(X,yy)

    stack_oof[:,j]=meta_oof
    stack_val[:,j]=final.predict_proba(Xv)[:,1]
    stack_test[:,j]=final.predict_proba(Xt)[:,1]

    stack_report.append({
        "label":label,
        "features":X.shape[1],
    })

pd.DataFrame(stack_report).to_csv(
    RUN_DIR/"stacker_report.csv",index=False
)
print("✅ cross-fitted stackers built")


In [ ]:

# 11. Controlled validation calibration
# For each label compare:
# - stacker
# - mE5
# - Twitter-XLM-R
# - lexical
# using only a small threshold/top-k menu.

source_triplets={
    "stack":(stack_oof,stack_val,stack_test),
    "me5":(me5_oof,me5_val,me5_test),
    "twitter":(tw_oof,tw_val,tw_test),
    "lex":(
        v29["lexical_oof"],
        v29["lexical_validation"],
        v29["lexical_test"],
    ),
}

selected={}
rows=[]

for j,label in enumerate(LABELS):
    best=None
    train_prior=float(train[label].mean())
    combined_prior=float(
        pd.concat([train[label],valid[label]],ignore_index=True).mean()
    )

    for sname,(po,pv,pt) in source_triplets.items():
        # Train threshold, applied unchanged to validation/test.
        t=best_threshold(label,y_train[:,j],po[:,j])

        policies=[
            ("threshold",t,None),
            ("topk",None,0.90),
            ("topk",None,1.00),
            ("topk",None,1.10),
        ]

        for policy,thr,scale in policies:
            if policy=="threshold":
                vpred=(pv[:,j]>=thr).astype(np.int8)
                tpred=(pt[:,j]>=thr).astype(np.int8)
            else:
                vpred=topk_pred(pv[:,j],train_prior*scale)
                tpred=topk_pred(pt[:,j],combined_prior*scale)

            vs=metric(label,y_valid[:,j],vpred)

            # Penalize large prevalence mismatch.
            prior_gap=abs(vpred.mean()-y_valid[:,j].mean())
            robust=vs-0.10*prior_gap

            rec={
                "label":label,
                "source":sname,
                "policy":policy,
                "threshold":thr,
                "scale":scale,
                "validation_score":float(vs),
                "validation_pred_rate":float(vpred.mean()),
                "validation_true_rate":float(y_valid[:,j].mean()),
                "robust":float(robust),
            }
            rows.append(rec)

            key=(robust,vs,-prior_gap)

            if best is None or key>best["key"]:
                best={
                    "key":key,
                    "rec":rec,
                    "validation_pred":vpred,
                    "test_pred":tpred,
                }

    selected[label]=best

selection=pd.DataFrame(rows)
selection.to_csv(
    RUN_DIR/"v30_validation_selection.csv",index=False
)

print("SELECTED:")
display(pd.DataFrame([selected[l]["rec"] for l in LABELS]).round(5))


In [ ]:

# 12. Final V30 validation score + submission gate
val_pred=np.column_stack(
    [selected[l]["validation_pred"] for l in LABELS]
)
test_pred=np.column_stack(
    [selected[l]["test_pred"] for l in LABELS]
)

vscore,label_scores=official(y_valid,val_pred)

report=pd.DataFrame({
    "label":LABELS,
    "validation_score":label_scores,
    "true_rate":y_valid.mean(0),
    "pred_rate":val_pred.mean(0),
    "test_pred_rate":test_pred.mean(0),
})
report.to_csv(
    RUN_DIR/"v30_final_validation_report.csv",index=False
)

SUBMIT_CANDIDATE=bool(
    vscore>=0.600
    and label_scores[0]>=0.70
)

summary={
    "version":"v30_me5_twitter_research_ensemble",
    "target":0.600,
    "validation_official_score":float(vscore),
    "toxic_macro_f1":float(label_scores[0]),
    "attribute_mean_f1":float(np.mean(label_scores[1:])),
    "SUBMIT_CANDIDATE":SUBMIT_CANDIDATE,
    "note":"Correct official split; no external labeled dataset; validation used for allowed tuning."
}

(RUN_DIR/"v30_summary.json").write_text(
    json.dumps(summary,indent=2)
)

sub=sample.copy()
for j,l in enumerate(LABELS):
    sub[l]=test_pred[:,j]

assert sub.ID.tolist()==test.ID.tolist()
assert sub[LABELS].isin([0,1]).all().all()

sub.to_csv(
    RUN_DIR/"submission_v30_target600.csv",index=False
)

print(json.dumps(summary,indent=2))
display(report.round(5))

if SUBMIT_CANDIDATE:
    print("\n✅ V30 PASSED THE 0.600 VALIDATION GATE.")
else:
    print("\n❌ V30 did not pass the gate. Do not submit before review.")


In [ ]:

# 13. Package V30 results
np.savez_compressed(
    RUN_DIR/"v30_new_probabilities.npz",
    me5_oof=me5_oof,me5_validation=me5_val,me5_test=me5_test,
    twitter_oof=tw_oof,twitter_validation=tw_val,twitter_test=tw_test,
    stack_oof=stack_oof,stack_validation=stack_val,stack_test=stack_test,
)

zip_path=Path("/content/v30_results.zip")

with zipfile.ZipFile(
    zip_path,"w",compression=zipfile.ZIP_DEFLATED
) as z:
    for p in RUN_DIR.iterdir():
        if p.is_file():
            z.write(p,arcname=p.name)

print("Created:",zip_path)

from google.colab import files
files.download(str(zip_path))
